# Week 5: Variational Autoencoder on Fashion-MNIST

I train VAEs, inspect reconstructions and generated images, then compare clustering with raw pixels, PCA and the Week 4 AE. Run the cells from top to bottom with the Python (research) kernel.

## 1. Imports, paths and device

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "week1" / "week1_ml_basics.ipynb").is_file()
                     and (folder / "week5" / "week5_vae.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter in the research project folder or one of its subfolders.")
WEEK5_ROOT = PROJECT_ROOT / "week5"
RESULTS_DIR = WEEK5_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROJECT_ROOT / "data"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


## 2. Dataset

The 20,000 training images and 10,000 test images match the Week 4 experiment setup.

In [ ]:
transform = transforms.ToTensor()
full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform)
full_test = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transform)
rng = np.random.default_rng(SEED)
train_indices = rng.choice(len(full_train), size=20000, replace=False)
test_indices = rng.choice(len(full_test), size=10000, replace=False)
train_set = Subset(full_train, train_indices)
test_set = Subset(full_test, test_indices)
train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)
X_test = full_test.data.numpy()[test_indices].astype(np.float32).reshape(-1, 784) / 255.0
y_test = full_test.targets.numpy()[test_indices]
print("Train:", len(train_set), "Test:", len(test_set))


## 3. VAE model

The encoder returns a mean and log variance. Reparameterization samples a latent vector while preserving the gradient path.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU()
        )
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid()
        )

    def encode(self, x):
        hidden = self.encoder(x)
        return self.fc_mu(hidden), self.fc_logvar(hidden)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

print(VAE(20))


## 4. VAE loss

The reconstruction term is binary cross entropy, measured per image. The KL term encourages the latent distribution to stay near a standard Gaussian.

In [ ]:
def vae_loss(reconstruction, images, mu, logvar):
    recon = nn.functional.binary_cross_entropy(reconstruction, images, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kl, recon, kl


## 5. Train and evaluate one VAE

In [ ]:
def train_vae(latent_dim, epochs=5):
    torch.manual_seed(SEED)
    model = VAE(latent_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    history = []
    for epoch in range(epochs):
        model.train()
        train_total = train_recon = train_kl = 0.0
        for images, _ in train_loader:
            images = images.to(device)
            optimizer.zero_grad()
            reconstruction, mu, logvar = model(images)
            loss, recon, kl = vae_loss(reconstruction, images, mu, logvar)
            loss.backward()
            optimizer.step()
            train_total += loss.item()
            train_recon += recon.item()
            train_kl += kl.item()
        model.eval()
        test_total = test_recon = test_kl = test_mean_recon = 0.0
        with torch.inference_mode():
            for images, _ in test_loader:
                images = images.to(device)
                reconstruction, mu, logvar = model(images)
                loss, recon, kl = vae_loss(reconstruction, images, mu, logvar)
                test_total += loss.item()
                test_recon += recon.item()
                test_kl += kl.item()
                mean_reconstruction = model.decode(mu)
                test_mean_recon += nn.functional.binary_cross_entropy(mean_reconstruction, images, reduction="sum").item()
        row = {
            "epoch": epoch + 1,
            "train_loss_per_image": train_total / len(train_set),
            "train_recon_per_image": train_recon / len(train_set),
            "train_kl_per_image": train_kl / len(train_set),
            "test_loss_per_image": test_total / len(test_set),
            "test_recon_bce_per_image": test_recon / len(test_set),
            "test_kl_per_image": test_kl / len(test_set),
            "test_mean_recon_bce_per_image": test_mean_recon / len(test_set),
        }
        history.append(row)
        print(f"d={latent_dim}, epoch {epoch + 1}: train={row['train_loss_per_image']:.2f}, test={row['test_loss_per_image']:.2f}")
    return model, history


## 6. Train four latent dimensions

The same settings are used for d = 2, 10, 20 and 50.

In [ ]:
latent_dimensions = [2, 10, 20, 50]
models = {}
histories = {}
for latent_dim in latent_dimensions:
    models[latent_dim], histories[latent_dim] = train_vae(latent_dim, epochs=5)
    torch.save(models[latent_dim].state_dict(), RESULTS_DIR / f"vae_d{latent_dim}.pt")
with (RESULTS_DIR / "vae_histories.json").open("w") as f:
    json.dump(histories, f, indent=2)


## 7. Plot training curves

In [ ]:
for latent_dim in latent_dimensions:
    plt.plot([row["epoch"] for row in histories[latent_dim]],
             [row["test_loss_per_image"] for row in histories[latent_dim]],
             marker="o", label=f"d={latent_dim}")
plt.xlabel("Epoch")
plt.ylabel("Test VAE loss per image")
plt.title("VAE training curves")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "vae_loss_curves.png", dpi=180)
plt.show()


## 8. Original images and VAE reconstructions

The first row has 10 test images. The second row has their reconstructions.

In [ ]:
model_20 = models[20].eval()
images, _ = next(iter(DataLoader(test_set, batch_size=10, shuffle=False)))
with torch.inference_mode():
    mu, logvar = model_20.encode(images.to(device))
    reconstructed = model_20.decode(mu).cpu()
fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for i in range(10):
    axes[0, i].imshow(images[i, 0], cmap="gray")
    axes[1, i].imshow(reconstructed[i, 0], cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("VAE")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "vae_reconstructions_10.png", dpi=180)
plt.show()


## 9. Generate 20 new images

These images are decoded from random z sampled from N(0, I). They are not reconstructions of test images.

In [ ]:
torch.manual_seed(SEED)
with torch.inference_mode():
    z = torch.randn(20, 20, device=device)
    generated = model_20.decode(z).cpu()
fig, axes = plt.subplots(4, 5, figsize=(7.5, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i, 0], cmap="gray")
    ax.axis("off")
fig.suptitle("20 images sampled from the VAE prior")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "vae_generated_20.png", dpi=180)
plt.show()


## 10. Extract the mean latent vectors

The mean mu is deterministic, so I use it for K-means.

In [ ]:
def extract_mu(model, loader):
    model.eval()
    pieces = []
    with torch.inference_mode():
        for images, _ in loader:
            mu, _ = model.encode(images.to(device))
            pieces.append(mu.cpu().numpy())
    return np.concatenate(pieces)

latent_features = {d: extract_mu(models[d], test_loader) for d in latent_dimensions}
np.save(RESULTS_DIR / "vae_mu_20.npy", latent_features[20])
print("Latent shape:", latent_features[20].shape)


## 11. Compare clustering methods

All methods use the same 10,000 test images and K-means settings. Week 4 AE-20 results are read from its saved experiment record.

In [ ]:
def cluster_scores(features):
    ids = KMeans(n_clusters=10, n_init=10, random_state=SEED).fit_predict(features)
    return normalized_mutual_info_score(y_test, ids), adjusted_rand_score(y_test, ids)

pca_features = PCA(n_components=20, random_state=SEED).fit_transform(X_test)
week4_results = json.loads((PROJECT_ROOT / "week4" / "results" / "ae_comparison.json").read_text())
ae_20 = next(row for row in week4_results if row.get("latent_dim") == 20)
rows = []
for name, features in [("Raw pixels", X_test), ("PCA-20", pca_features), ("VAE-20", latent_features[20])]:
    nmi, ari = cluster_scores(features)
    rows.append({"Representation": name, "Dimension": features.shape[1], "NMI": nmi, "ARI": ari})
rows.insert(2, {"Representation": "AE-20", "Dimension": 20, "NMI": ae_20["NMI"], "ARI": ae_20["ARI"]})
comparison = pd.DataFrame(rows)
comparison.to_csv(RESULTS_DIR / "representation_comparison.csv", index=False)
print(comparison.to_string(index=False))


## 12. Compare VAE latent dimensions

In [ ]:
dimension_rows = []
for latent_dim in latent_dimensions:
    nmi, ari = cluster_scores(latent_features[latent_dim])
    dimension_rows.append({
        "latent_dim": latent_dim,
        "test_vae_loss_per_image": histories[latent_dim][-1]["test_loss_per_image"],
        "test_recon_bce_per_image": histories[latent_dim][-1]["test_recon_bce_per_image"],
        "test_kl_per_image": histories[latent_dim][-1]["test_kl_per_image"],
        "test_mean_recon_bce_per_image": histories[latent_dim][-1]["test_mean_recon_bce_per_image"],
        "NMI": nmi,
        "ARI": ari,
    })
dimension_comparison = pd.DataFrame(dimension_rows)
dimension_comparison.to_csv(RESULTS_DIR / "vae_dimension_comparison.csv", index=False)
print(dimension_comparison.to_string(index=False))


## 13. Visualize the two-dimensional latent space

In [ ]:
z2 = latent_features[2]
fig, ax = plt.subplots(figsize=(7, 5.5))
scatter = ax.scatter(z2[:, 0], z2[:, 1], c=y_test, s=3, cmap="tab10", alpha=0.7)
ax.set(xlabel="mu 1", ylabel="mu 2", title="VAE latent means (d=2)")
fig.colorbar(scatter, ax=ax, ticks=range(10), label="Fashion-MNIST class")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "vae_latent_2.png", dpi=180)
plt.show()


## 14. Interpolate between two test images

The endpoints are fixed test samples from two different classes. Intermediate mu vectors are decoded into images.

In [ ]:
first_index = 0
second_index = next(i for i in range(1, len(y_test)) if y_test[i] == 0)
endpoints = torch.tensor(np.stack([X_test[first_index], X_test[second_index]]).reshape(2, 1, 28, 28), device=device)
with torch.inference_mode():
    endpoint_mu, _ = model_20.encode(endpoints)
    alphas = torch.linspace(0, 1, 9, device=device).unsqueeze(1)
    path = (1 - alphas) * endpoint_mu[0] + alphas * endpoint_mu[1]
    interpolated = model_20.decode(path).cpu()
fig, axes = plt.subplots(1, 9, figsize=(12, 1.8))
for i, ax in enumerate(axes):
    ax.imshow(interpolated[i, 0], cmap="gray")
    ax.axis("off")
    ax.set_title(f"{i/8:.2f}", fontsize=8)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "vae_interpolation.png", dpi=180)
plt.show()
print("Endpoint labels:", y_test[first_index], "and", y_test[second_index])


## My observations

I compare the NMI and ARI values before deciding whether VAE features help clustering. The VAE loss combines reconstruction and KL terms, so I do not compare its numerical value directly with the AE MSE loss. Reconstruction uses an input image; generation starts from a random latent sample. Interpolation shows how the decoder changes an image as the latent vector moves between two examples.